# 01 — EDA: Label Distribution

Exploration only — no logic lives here (see `Architecture.md`). Source data: `data/processed/{train,val,test}.parquet`, produced by `python -m src.data.preprocess` (Phase 1).

Dataset = Jigsaw Toxic Comment Classification Challenge (2018) + Civil Comments (`jigsaw-unintended-bias-in-toxicity-classification`, subtype scores binarized at 0.5 and mapped onto the Jigsaw 2018 label schema; `identity_attack` -> `identity_hate`). Deduplicated on exact comment text, capped at 300,000 rows via stratified sampling on the rarest label (`severe_toxic`), split 80/10/10 stratified on the same key. See `Memory.md` for the full decision log.

In [1]:
import json

import pandas as pd

with open("../configs/labels.json") as f:
    LABELS = json.load(f)["labels"]

splits = {name: pd.read_parquet(f"../data/processed/{name}.parquet") for name in ["train", "val", "test"]}

In [2]:
for name, df in splits.items():
    print(f"{name}: {len(df)} rows")
    for label in LABELS:
        print(f"  {label:<14} {df[label].mean() * 100:.2f}%")
    print()

train: 240000 rows
  toxic          8.12%
  severe_toxic   0.08%
  obscene        0.92%
  threat         0.26%
  insult         5.85%
  identity_hate  0.77%

val: 30000 rows
  toxic          8.27%
  severe_toxic   0.08%
  obscene        0.93%
  threat         0.24%
  insult         5.97%
  identity_hate  0.73%

test: 30000 rows
  toxic          8.04%
  severe_toxic   0.08%
  obscene        0.86%
  threat         0.21%
  insult         5.77%
  identity_hate  0.74%


Positive rates are consistent across splits (stratification on `severe_toxic` worked as intended and the other five labels came along for the ride, since they're all correlated with overall toxicity). `severe_toxic` and `threat` are the rarest labels by a wide margin — this is the class-imbalance signal that motivates the weighted-BCE-loss decision in Phase 3 (see `Phases.md`, `Rules.md`).

In [ ]:
import matplotlib.pyplot as plt

train_rates = splits["train"][LABELS].mean().sort_values(ascending=True) * 100

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(train_rates.index, train_rates.values)
ax.set_xlabel("Positive rate (%)")
ax.set_title("Label distribution \u2014 train split (n=240,000)")
fig.tight_layout()
plt.show()